# FYP — Environment Check

**Project:** Sarcasm and Irony Detection in Vietnamese Social Media Using Transformer-Based Models

**Purpose:** Verify the Python environment, project dependencies, and NVIDIA GPU access before running data collection or model experiments.

- This notebook **does not install packages**, download models, or modify files.
- It never displays API keys or private dataset contents.
- Use the **Python (FYP Sarcasm)** Jupyter kernel.
- Results may vary by machine; do not treat the checks as model evaluation.

> **Before committing to GitHub:** Use **Kernel → Restart & Run All** to check locally, then **Edit → Clear All Outputs** and save the notebook. This avoids publishing local environment information in saved output.

## 1. Python environment

Display only version information and the environment **name** (not an absolute user-specific path).

In [ ]:
import sys
import platform
from pathlib import Path

print('OS:', platform.system())
print('Python:', sys.version.split()[0])
print('Environment:', Path(sys.prefix).name)
print('Python architecture:', platform.architecture()[0])

if Path(sys.prefix).name.lower() != 'fyp-sarcasm':
    print('NOTE: Verify that the selected kernel belongs to the fyp-sarcasm environment.')


## 2. Project dependency check

Report installed package versions **without importing every library**. This keeps the check lightweight and avoids unnecessary side effects. `Optional` packages are useful later but do not block early model development.

In [ ]:
from importlib import util, metadata

REQUIRED = {
    # Data and traditional machine learning
    'numpy': 'numpy',
    'pandas': 'pandas',
    'scipy': 'scipy',
    'scikit-learn': 'sklearn',
    'matplotlib': 'matplotlib',
    # GPU training / Transformers
    'torch': 'torch',
    'transformers': 'transformers',
    'datasets': 'datasets',
    'accelerate': 'accelerate',
    'sentencepiece': 'sentencepiece',
    # Vietnamese comments and the emoji ablation
    'underthesea': 'underthesea',
    'emoji': 'emoji',
    # Data collection
    'google-api-python-client': 'googleapiclient',
    'python-dotenv': 'dotenv',
    'requests': 'requests',
    'beautifulsoup4': 'bs4',
    'lxml': 'lxml',
    # Annotation analysis and utilities
    'openpyxl': 'openpyxl',
    'tqdm': 'tqdm',
}

OPTIONAL = {
    'jupyterlab': 'jupyterlab',
    'streamlit': 'streamlit',  # Demo phase
    'protobuf': 'google.protobuf',
}


def package_status(dist_name, module_name):
    try:
        present = util.find_spec(module_name) is not None
    except (ImportError, ModuleNotFoundError, ValueError):
        present = False
    if not present:
        return False, 'NOT FOUND'
    try:
        return True, metadata.version(dist_name)
    except metadata.PackageNotFoundError:
        return True, 'version unavailable'


missing_required = []
print('REQUIRED PACKAGES')
for package, module in REQUIRED.items():
    found, ver = package_status(package, module)
    if not found:
        missing_required.append(package)
    print(f"{'OK' if found else 'MISSING':7} {package:<28} {ver}")

print('\nOPTIONAL PACKAGES')
for package, module in OPTIONAL.items():
    found, ver = package_status(package, module)
    print(f"{'OK' if found else 'OPTIONAL':8} {package:<28} {ver}")


## 3. PyTorch and CUDA

Verify GPU visibility and perform a **small CUDA matrix calculation**. The expected device on the development laptop is an NVIDIA GeForce RTX 3060 Laptop GPU (6 GB VRAM). No large tensor allocations or model downloads are performed.

In [ ]:
gpu_available = False
gpu_test_passed = False

try:
    import torch

    print('PyTorch:', torch.__version__)
    print('PyTorch CUDA runtime:', torch.version.cuda)
    gpu_available = bool(torch.cuda.is_available())
    print('CUDA available:', gpu_available)

    if gpu_available:
        print('GPU:', torch.cuda.get_device_name(0))
        props = torch.cuda.get_device_properties(0)
        print('VRAM: {:.2f} GB'.format(props.total_memory / (1024 ** 3)))

        a = torch.ones((128, 128), device='cuda')
        b = a @ a
        torch.cuda.synchronize()
        gpu_test_passed = bool(torch.isfinite(b).all().item())
        print('Small CUDA operation:', 'PASS' if gpu_test_passed else 'FAIL')
        del a, b
        torch.cuda.empty_cache()
    else:
        print('GPU test: SKIPPED (no CUDA device available)')
except Exception as exc:
    print('PyTorch / CUDA check failed:', type(exc).__name__, str(exc))


## 4. Core FYP functionality

Check the imports needed for the **TF-IDF + Logistic Regression** baseline, **Cohen's kappa**, and **PhoBERT / XLM-R** pipelines. This is an import smoke test, **not** model training.

In [ ]:
core_imports_passed = True

checks = {
    'Baseline and metrics': lambda: __import__('sklearn.feature_extraction.text', fromlist=['TfidfVectorizer']).TfidfVectorizer,
    'Logistic regression': lambda: __import__('sklearn.linear_model', fromlist=['LogisticRegression']).LogisticRegression,
    'Cohen kappa': lambda: __import__('sklearn.metrics', fromlist=['cohen_kappa_score']).cohen_kappa_score,
    'Hugging Face models': lambda: __import__('transformers', fromlist=['AutoTokenizer']).AutoTokenizer,
    'YouTube API client': lambda: __import__('googleapiclient.discovery', fromlist=['build']).build,
}

for name, loader in checks.items():
    try:
        loader()
        print(f'PASS  {name}')
    except Exception as exc:
        core_imports_passed = False
        print(f'FAIL  {name}: {type(exc).__name__}: {exc}')


## 5. Summary

For transformer fine-tuning, you need the required dependencies, valid imports, and a working CUDA GPU. The environment check does **not** test model accuracy. A separate model smoke test can load PhoBERT and XLM-R **one at a time** if needed.

In [ ]:
print('Required packages:', 'PASS' if not missing_required else 'MISSING: ' + ', '.join(missing_required))
print('Core imports:', 'PASS' if core_imports_passed else 'FAIL')
print('CUDA detection:', 'PASS' if gpu_available else 'FAIL')
print('CUDA calculation:', 'PASS' if gpu_test_passed else 'FAIL')

all_ready = not missing_required and core_imports_passed and gpu_test_passed
print('\nOVERALL:', 'READY FOR FYP EXPERIMENTS' if all_ready else 'SOME CHECKS NEED ATTENTION')


## Notes for reproduction

- Create or activate the `fyp-sarcasm` Conda environment before launching JupyterLab.
- Use the CUDA-enabled PyTorch installation command documented in your repository `README.md`. A plain `pip freeze` may not communicate the correct PyTorch CUDA index to a new computer.
- Keep `.env` (YouTube API credentials), raw data, and trained model checkpoints outside Git history.
- Clear notebook outputs before pushing updates to GitHub.